# SmartClean Twin — Project Walkthrough & Discussion

**Course:** RBB2013 Digital Twin — May 2026
**Student:** Chan Li Kai (22010900) — group project
**Purpose:** live walkthrough of the complete Digital Twin for discussion —
review what has been built and identify anything worth adding before submission (19 July).

> Prerequisite: `docker compose up -d` running (8 containers).


## 1. Project at a Glance

Digital Twin of a **mobile cleaning robot** (topic 2 from the project list).

| Layer | Technology | What it does |
|---|---|---|
| Physical asset (simulated) | Python physics simulator | 5m x 5m room, lawnmower path, battery + charging cycle, fault injection |
| Streaming | MQTT (Mosquitto), port 1883 | Telemetry / state / commands, JSON, 1 msg/s |
| Validation | telemetry-ingestion service | Pydantic schema validation before storage |
| Storage | InfluxDB 2.7, port 8086 | Time-series persistence (telemetry, state, predictions, alerts) |
| Twin state | state-engine service | 11-dimension state model (safety, mission, battery, ...) |
| AI | ai-service — 5 models, 3 ML paradigms | Health classification, RUL regression, anomaly detection + forecasts |
| Control | command-api, port 8000 | REST → MQTT commands with acknowledgements |
| Visualization | Grafana (28 panels, 7 sections) + NVIDIA Omniverse 3D | Operator dashboard + live 3D twin |

All services individually containerized (8 containers), with unit / integration /
system / regression test suites and CI on GitHub Actions.


## 2. Architecture

```
 Robot Simulator ──raw──> Mosquitto MQTT <──commands── Command API (REST :8000)
        │                    │      ▲
        │ (1s physics tick)  │      └── acks
        ▼                    ▼
                   Telemetry Ingestion ──validated──> InfluxDB :8086
                             │                           │
                             ▼                           ▼
                        State Engine ──state──>      Grafana :3001
                             │                           ▲
                             ▼                           │
                         AI Service ──predictions────────┘
                     (:8003, 5 models + /whatif)
                                          Omniverse 3D (polls InfluxDB, 1s)
```

Full interface contract (every topic, port, payload schema): `docs/api-contract.md`.


## 3. Live System Check
All 5 application services should answer their health endpoints.

In [ ]:
import json, urllib.request

SERVICES = {
    "command-api": "http://localhost:8000/health",
    "telemetry-ingestion": "http://localhost:8001/health",
    "state-engine": "http://localhost:8002/health",
    "ai-service": "http://localhost:8003/health",
    "robot-simulator": "http://localhost:8004/health",
}
for name, url in SERVICES.items():
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            h = json.loads(r.read())
            print(f"{name:22s} OK   uptime={h.get('uptime_s', '?')}s")
    except Exception as e:
        print(f"{name:22s} DOWN ({e})")


## 4. Live Telemetry from InfluxDB
Most recent sensor readings stored by the pipeline (updates every second — re-run the cell).

In [ ]:
INFLUX = "http://localhost:8086/api/v2/query?org=smartclean"
TOKEN = "smartclean-super-secret-token"

def flux_query(q):
    req = urllib.request.Request(
        INFLUX, data=q.encode(),
        headers={"Authorization": f"Token {TOKEN}",
                 "Content-Type": "application/vnd.flux",
                 "Accept": "application/csv"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return r.read().decode()

def show_last(measurement):
    q = (f'from(bucket: "smartclean_twin") |> range(start: -30s) '
         f'|> filter(fn: (r) => r._measurement == "{measurement}") |> last()')
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 7 and p[1] == "_result":
            print(f"  {p[7]:28s} = {p[6]}")

print("robot_telemetry:")
show_last("robot_telemetry")


## 5. AI Layer — 5 Models, 3 ML Paradigms

| Model | Paradigm | Performance | Predicts |
|---|---|---|---|
| Motor health classifier | Supervised classification | 100% test acc | NORMAL / HIGH_LOAD / OVERHEATED / FAULT |
| Dirt level classifier | Supervised classification | 99.9% | CLEAN / MODERATE / DIRTY |
| Health state classifier | Supervised classification | 90.8% | NORMAL / WARNING / CRITICAL |
| RUL regressor | Supervised regression | R²=0.91, MAE 6.6 min | Remaining useful life (minutes) |
| Anomaly detector | **Unsupervised** | 100% fault detection, 0% false alarms | Never-seen sensor patterns (learned from normal operation only) |

Plus live trend-based forecasts (battery minutes-to-empty, cleaning minutes-to-finish)
and an **operator recommendation** combining all outputs — the twin advises, not just monitors.

Methodology: labelled synthetic datasets with documented physics-motivated rules,
80/20 stratified split, StandardScaler pipelines, cross-checked metrics, models
trained at Docker build time (baked into the image, reproducible).


### 5b. Latest live prediction

In [ ]:
print("robot_prediction:")
show_last("robot_prediction")


## 6. What-If Simulation (core Digital Twin capability)
Test hypothetical scenarios against all 5 models **without touching the robot**.

In [ ]:
def whatif(**scenario):
    req = urllib.request.Request(
        "http://localhost:8003/whatif",
        data=json.dumps(scenario).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())["prediction"]

for label, scenario in [
    ("A: healthy robot", dict(motor_temperature_c=40, motor_current_a=0.8, battery_soc=90)),
    ("B: overheating under load", dict(motor_temperature_c=90, motor_current_a=3.6, battery_soc=40)),
    ("C: low battery + low water", dict(battery_soc=15, water_level_pct=5)),
]:
    p = whatif(**scenario)
    print(f"Scenario {label}")
    print(f"  health={p['health_state_prediction']}  "
          f"RUL={p['predicted_rul_minutes']} min  anomaly={p['is_anomaly']}")
    print(f"  -> {p['recommendation']}")
    print()


## 7. Fault Injection Demo (run live)
Injects a motor overload — watch the Grafana overview strip react
(http://localhost:3001/d/smartclean-main), then clears the fault.

In [ ]:
import time

def inject(fault):
    req = urllib.request.Request(
        "http://localhost:8004/fault",
        data=json.dumps({"fault": fault}).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        print(f"POST /fault {fault!r} -> HTTP {r.status}")

inject("motor")
print("Motor overload injected — watch the dashboard overview strip ...")
time.sleep(20)
q = ('from(bucket: "smartclean_twin") |> range(start: -15s) '
     '|> filter(fn: (r) => r._measurement == "robot_prediction" and '
     '(r._field == "recommendation" or r._field == "anomaly_score" '
     'or r._field == "health_state")) |> last()')
for line in flux_query(q).splitlines():
    p = line.split(",")
    if len(p) > 7 and p[1] == "_result":
        print(f"  {p[7]} = {p[6]}")
inject("clear")
print("Fault cleared — dashboard returns to green within ~30 s.")


## 8. Development Practices Evidence

- **Sprints:** 2 documented cycles — features, milestones, reviews (`docs/sprint-plan.md`)
- **Tests:** 81 unit tests + integration + system + regression suites; pass AND fail cases demonstrated
- **CI/CD:** GitHub Actions — ruff + black lint, full test suite, Docker build on every push
- **Version control:** consistent commit history; teammate review docs pushed from their own accounts
- **Persistence:** proven across container restart (`tests/system/test_persistence.py` — 3/3 pass)
- **Scaling:** `docker compose up --scale telemetry-ingestion=2` (safe — MQTT subscription fan-out)


## 9. Beyond the Rubric

- **NVIDIA Omniverse 3D twin** — live USD scene: robot pose + heading arrow, coverage tiles,
  battery bar, safety-state colours, flashing EMERGENCY light, obstacle indicator, breadcrumb trail
- **Autonomous battery lifecycle** — returns home below 20%, charges at dock, resumes cleaning
- **What-if API** — scenario testing endpoint (section 6)
- **AI recommendation banner** — the twin advises the operator (decision loop closed)


## 10. Discussion — Questions for Dr.

1. **Scope:** does the current depth match expectations for the 40% project?
2. **AI direction:** we cover 3 paradigms (classification / regression / unsupervised anomaly).
   Would a time-series forecasting model (e.g. temperature trend prediction) add value, or is breadth sufficient?
3. **Multi-robot:** topics already carry robot_id (`smartclean/SCR01/...`).
   Worth demonstrating a second robot instance, or out of scope?
4. **Security:** MQTT/REST currently unauthenticated (documented limitation).
   Add TLS/auth, or is documenting it acceptable at this level?
5. **Presentation order:** architecture first or live demo first?
6. **Peer review:** any preferred format for the individual review documents?

### Known limitations (honest list)
- Robot is simulated — swap-in point for real hardware is the simulator container only; nothing else changes
- Models trained on synthetic data with documented labelling rules (correct methodology, generated data)
- Single-host docker compose (no Kubernetes/orchestrator)
- No authentication on MQTT / REST APIs
